# Groww ingestion walkthrough (Nifty50 & HDFC.NS)This notebook exercises the QC-Engine ingestion stack end-to-end for two India-focused assets:- **Nifty50** (index-level benchmark)- **HDFC Bank (HDFC.NS)**It wires up the Groww adapter/client (with optional credentials), backfills 10 years of daily candles into the Parquet store, and renders quick plots to confirm the pipeline behaves as expected. The same flow is run with the built-in yfinance adapter to provide a working data source even when Groww credentials are unavailable.

## Prerequisites- Python environment with the project installed (``pip install -e .`` from the repo root).- Optional Groww credentials exposed as environment variables:  - ``GROWW_ACCESS_TOKEN`` **or** ``GROWW_API_KEY`` (plus ``GROWW_API_SECRET`` when required).- Network access for yfinance/Groww HTTP calls.> The notebook keeps imports within the QC-Engine SDK boundary (adapters carry all provider-specific dependencies).

In [ ]:
from __future__ import annotationsimport osfrom datetime import datetime, timedelta, timezonefrom pathlib import Pathimport pandas as pdfrom qcengine.adapters.base import InstrumentReffrom qcengine.adapters.groww import GrowwClient, GrowwClientConfig, GrowwMarketDataAdapterfrom qcengine.adapters.yfinance import YFinanceMarketDataAdapterfrom qcengine.analytics.plotting import candles_to_dataframe, plot_candles_matplotlibfrom qcengine.domain.marketdata import Timeframefrom qcengine.ingestion.backfill import BackfillJobfrom qcengine.storage.parquet_store import ParquetCandleStore

## Configure instruments and time rangeWe target a **decade** of daily bars for two instruments. ``instrument_id`` is the canonical key stored in Parquet; ``provider_symbol`` is specific to each adapter. Groww also requires ``exchange`` and ``segment``—fill these with the exact values used by your Groww account.

In [ ]:
# Define the 10-year window (UTC aware)end_utc = datetime.now(timezone.utc)start_utc = end_utc - timedelta(days=365 * 10)timeframe = Timeframe.DAY_1# Canonical instrument identitiesnifty50 = InstrumentRef(    instrument_id="NIFTY50",    provider_symbol="^NSEI",  # yfinance symbol; update provider_symbol/exchange/segment for Groww    exchange="NSE",    segment="INDICES",)hdfc_bank = InstrumentRef(    instrument_id="HDFC_BANK",    provider_symbol="HDFC.NS",    exchange="NSE",    segment="EQUITY",)instruments = [nifty50, hdfc_bank]print(f"Time window: {start_utc.date()} -> {end_utc.date()} ({timeframe.value})")

## Initialize Parquet storageThe Parquet store handles partitioning and deduplication keyed by ``(instrument_id, timeframe, bar_start_ts_utc)``. Storing under ``scripts/data`` keeps demo artifacts alongside the notebook.

In [ ]:
store_root = Path("scripts/data/groww_ingestion")store = ParquetCandleStore(store_root)store_root

## Build adapters- **yfinance**: Always available and used to validate the ingestion pipeline.- **Groww**: Enabled when credentials are supplied; otherwise skipped. The client will auto-refresh an access token when ``api_key``/``api_secret`` are present.

In [ ]:
# yfinance adapter (no configuration required)yf_adapter = YFinanceMarketDataAdapter()# Optional Groww adapteraccess_token = os.getenv("GROWW_ACCESS_TOKEN")api_key = os.getenv("GROWW_API_KEY")api_secret = os.getenv("GROWW_API_SECRET")use_groww = bool(access_token or api_key)print(f"Groww enabled: {use_groww}")if use_groww:    groww_config = GrowwClientConfig(        api_key=api_key,        api_secret=api_secret,        access_token=access_token,    )    groww_client = GrowwClient(groww_config)    groww_adapter = GrowwMarketDataAdapter(groww_client)else:    groww_adapter = None

## Backfill via yfinanceRunning the backfill with yfinance ensures we have data to exercise storage + plotting even when Groww is unavailable. The job is idempotent—re-running will deduplicate based on the canonical candle key.

In [ ]:
yf_job = BackfillJob(    adapter=yf_adapter,    storage=store,    instruments=instruments,    timeframes=[timeframe],    start_utc=start_utc,    end_utc=end_utc,)yf_job.run()

## (Optional) Overlay with Groww dataWhen credentials are present, this cell fetches the same window from Groww. Duplicate rows are automatically coalesced in Parquet, letting you compare provider outputs or prefer one source.

In [ ]:
if groww_adapter is not None:    groww_job = BackfillJob(        adapter=groww_adapter,        storage=store,        instruments=instruments,        timeframes=[timeframe],        start_utc=start_utc,        end_utc=end_utc,    )    groww_job.run()else:    print("Groww credentials not supplied; skipping Groww backfill.")

## Validate storage contentsWe load the persisted candles, confirm ordering, and compute quick summary stats to ensure the ingestion pipeline behaved as expected.

In [ ]:
def summarize(instrument: InstrumentRef):    candles = store.read(instrument.instrument_id, timeframe, start_utc, end_utc)    df = candles_to_dataframe(candles)    print(        f"{instrument.instrument_id}: {len(candles)} candles, "        f"range {df.index.min().date()} -> {df.index.max().date()}"    )    assert df.index.is_monotonic_increasing, "Candles must be time-sorted"    return candles, dfnifty_candles, nifty_df = summarize(nifty50)hdfc_candles, hdfc_df = summarize(hdfc_bank)

## Plot a decade of candlesMatplotlib plots offer a quick visual confirmation that prices and volumes were ingested correctly. Adjust the rendering backend as needed for your environment (e.g., `%matplotlib inline` in classic Jupyter).

In [ ]:
_ = plot_candles_matplotlib(nifty_candles, annotate_timeframe=True, show=True)_ = plot_candles_matplotlib(hdfc_candles, annotate_timeframe=True, show=True)